In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
from prophet import Prophet
import stan


In [ ]:
!sudo apt-get update
!sudo apt-get install -y build-essential

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
df = pd.read_parquet('/content/rent_mortgage_all_features.parquet').fillna(0)
df

,RegionID,RegionName,StateName,Date,HomeValue,IncomeNeeded,Inventory,DaysToPending,RentValue,RenterIncomeNeeded,...,cpi_lag1,cpi_roll3,mortgage_rate_lag1,mortgage_rate_roll3,unrate_lag1,unrate_roll3,MortgageValue,Rent_to_Mortgage_Ratio,MortgageValue_Change,Sales_to_Inventory
0,394304,"Akron, OH",OH,2018-05-01,141498.445820,37039.911190,3632.0,58.0,842.862503,33714.500133,...,0.000,0.000000,0.0000,0.000000,0.0,0.000000,578.230936,1.457657,0.000000,0.294879
1,394304,"Akron, OH",OH,2018-06-01,141997.352756,37126.302161,3806.0,55.0,844.909823,33796.392940,...,250.792,0.000000,4.5860,0.000000,3.8,0.000000,579.186971,1.458786,0.001653,0.254073
2,394304,"Akron, OH",OH,2018-07-01,142509.730768,37144.015612,3917.0,54.0,844.722835,33788.913391,...,251.018,251.008000,4.5700,4.561167,4.0,3.866667,578.395359,1.460459,-0.001367,0.266786
3,394304,"Akron, OH",OH,2018-08-01,143107.271515,37358.420177,3968.0,53.0,846.884637,33875.385472,...,251.214,251.298333,4.5275,4.549167,3.8,3.866667,582.351584,1.454250,0.006840,0.257812
4,394304,"Akron, OH",OH,2018-09-01,143845.022393,37757.841397,3918.0,56.0,850.004092,34000.163688,...,251.663,251.686333,4.5500,4.568333,3.8,3.766667,590.669795,1.439051,0.014284,0.204952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7999,395238,"Worcester, MA",MA,2025-03-01,471121.892980,128746.065420,1315.0,33.0,2118.639336,84745.573448,...,319.775,319.492000,6.8425,6.816833,4.1,4.100000,2406.662822,0.880322,-0.018957,0.461597
8000,395238,"Worcester, MA",MA,2025-04-01,471093.632963,129493.319360,1495.0,26.0,2125.944172,85037.766887,...,319.615,319.903667,6.6500,6.739167,4.2,4.166667,2425.157969,0.876621,0.007685,0.496321
8001,395238,"Worcester, MA",MA,2025-05-01,470533.524469,130264.921899,1799.0,21.0,2130.885088,85235.403528,...,320.321,320.172000,6.7250,6.730333,4.2,4.200000,2444.942673,0.871548,0.008158,0.501946
8002,395238,"Worcester, MA",MA,2025-06-01,469919.352005,130117.860196,2037.0,20.0,2140.907753,85636.310100,...,320.580,320.800333,6.8160,6.786167,4.2,4.166667,2442.125250,0.876658,-0.001152,0.534610


In [ ]:
%pip install pystan

In [ ]:
def forecast_metro_prophet(df, regressors, target, forecast_months=24):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df = df.rename(columns={"Date": "ds", target: "y"}).dropna(subset=["y"])
    numeric_cols = [c for c in numeric_cols if c != target]
    df = df.set_index("ds").resample("ME")[numeric_cols + ["y"]].mean().reset_index()

    model = Prophet(yearly_seasonality=True, stan_backend='pystan')
    for reg in regressors:
        model.add_regressor(reg)

    model.fit(df[["ds", "y"] + regressors])

    # Create 24-month future dataframe
    future = model.make_future_dataframe(periods=forecast_months, freq="ME")

    # Merge known regressors (for past) with NaN for future
    future = future.merge(df[["ds"] + regressors], on="ds", how="left")

    # Forward fill NaN values in regressor columns for future dates
    future.fillna(method='ffill', inplace=True)


    forecast = model.predict(future)

    return forecast

In [ ]:
rent_regressors = [
    "HomeValue",       # property value influences rent
    "Inventory",       # supply pressure
    "SalesCount",      # market activity
    "MarketHeatIndex", # local market trend
    "cpi",             # inflation
    "unrate",          # unemployment
    "mortgage_rate",   # mortgage rates can indirectly affect rent
    "MortgageBurden",  # helps capture cost-of-living influence
]
forecast_results = []

for region, df_region in df.groupby("RegionID"):
    if len(df_region) < 24:
        continue
    forecast = forecast_metro_prophet(df_region, regressors=rent_regressors, target="RentValue", forecast_months=64)
    forecast_results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "Forecast": forecast
    })

forecast_results_df = pd.DataFrame(forecast_results)

ValueError: Unknown stan backend: pystan

In [ ]:
# Create an empty list to store the data for the new dataframe
forecasted_values = []

# Iterate through each row in the forecast_results_df
for index, row in forecast_results_df.iterrows():
    region_id = row['RegionID']
    region_name = row['RegionName']
    state_name = row['StateName']
    forecast_df = row['Forecast']

    # Extract the 'yhat' column from the forecast dataframe
    yhat_values = forecast_df[['ds', 'yhat']].rename(columns={'ds': 'Date', 'yhat': 'ForecastedRentValue'})

    # Add region information to the yhat values
    yhat_values['RegionID'] = region_id
    yhat_values['RegionName'] = region_name
    yhat_values['StateName'] = state_name

    # Append to the list
    forecasted_values.append(yhat_values)

# Concatenate all the dataframes in the list
final_forecast_df = pd.concat(forecasted_values, ignore_index=True)

# Reorder columns
final_forecast_df = final_forecast_df[['RegionID', 'RegionName', 'StateName', 'Date', 'ForecastedRentValue']]
final_forecast_df

In [ ]:
final_forecast_df["Date"] = pd.to_datetime(final_forecast_df["Date"])

In [ ]:
def plotForecast(target, df):
  plt.figure(figsize=(10,6))
  national_trend = df.groupby("Date")[target].mean()
  plt.plot(national_trend.index, national_trend.values, linewidth=2)
  plt.title(f"National Trend: {target} (All Metros)")
  plt.xlabel("Year")
  plt.ylabel(f"Average {target} Needed ($)")
  plt.grid(True, linestyle='--', alpha=0.5)
  plt.tight_layout()
  plt.show()

In [ ]:
def plotTop10(target, df):
  latest_date = df["Date"].max()
  top10 = df[df["Date"] == latest_date].sort_values(target, ascending=False).head(10)

  plt.figure(figsize=(10,6))
  sns.barplot(y="RegionName", x=target, data=top10, palette="Reds_r")
  plt.title(f"Top 10 Metros by {target} ({latest_date.date()})")
  plt.xlabel(f"{target} Needed ($)")
  plt.ylabel("")
  plt.tight_layout()
  plt.show()

In [ ]:
plotForecast("ForecastedRentValue", final_forecast_df)

In [ ]:
plotTop10("ForecastedRentValue", final_forecast_df)

In [ ]:
mortgage_regressors = [
    "HomeValue",      # biggest driver of mortgage payment
    "MortgageBurden", # cost sensitivity to rates
    "cpi",            # macroeconomic factor
    "mortgage_rate",  # interest rate
    "SalesCount",     # optional market activity signal
    "Inventory",      # supply factor
    "MarketHeatIndex" # optional, trend indicator
]
forecast_mortgage_results = []

for region, df_region in df.groupby("RegionID"):
    if len(df_region) < 24:
        continue
    forecast = forecast_metro_prophet(df_region, regressors=mortgage_regressors, target="MortgageValue", forecast_months=64)
    forecast_mortgage_results.append({
        "RegionID": region,
        "RegionName": df_region["RegionName"].iloc[0],
        "StateName": df_region["StateName"].iloc[0],
        "Forecast": forecast
    })

forecast_mortgage_df = pd.DataFrame(forecast_mortgage_results)

In [ ]:
# Create an empty list to store the data for the new dataframe
forecasted_mortgage_values = []

# Iterate through each row in the forecast_results_df
for index, row in forecast_mortgage_df.iterrows():
    region_id = row['RegionID']
    region_name = row['RegionName']
    state_name = row['StateName']
    forecast_df = row['Forecast']

    # Extract the 'yhat' column from the forecast dataframe
    yhat_values = forecast_df[['ds', 'yhat']].rename(columns={'ds': 'Date', 'yhat': 'ForecastedMortgageValue'})

    # Add region information to the yhat values
    yhat_values['RegionID'] = region_id
    yhat_values['RegionName'] = region_name
    yhat_values['StateName'] = state_name

    # Append to the list
    forecasted_mortgage_values.append(yhat_values)

# Concatenate all the dataframes in the list
final_forecast_mortgage_df = pd.concat(forecasted_mortgage_values, ignore_index=True)

# Reorder columns
final_forecast_mortgage_df = final_forecast_mortgage_df[['RegionID', 'RegionName', 'StateName', 'Date', 'ForecastedMortgageValue']]
final_forecast_mortgage_df

In [ ]:
plotForecast("ForecastedMortgageValue", final_forecast_mortgage_df)

In [ ]:
plotTop10("ForecastedMortgageValue", final_forecast_mortgage_df)

In [ ]:
final_df = pd.merge(final_forecast_df, final_forecast_mortgage_df, on=["RegionID", "RegionName", "StateName", "Date"], how="inner")
final_df

In [ ]:
final_df["Rent_to_Mortgage_Ratio"] = final_df["ForecastedRentValue"] / final_df["ForecastedMortgageValue"]
final_df["Monthly_Savings_or_Cost"] = final_df["ForecastedMortgageValue"] - final_df["ForecastedRentValue"]


In [ ]:
final_df["Affordability_Index"] = final_df["ForecastedMortgageValue"] / (final_df["ForecastedRentValue"] + final_df["ForecastedMortgageValue"])

In [ ]:
# Average by region
region_summary = (
    final_df.groupby(["RegionName", "StateName"])
      .agg({
          "ForecastedRentValue": "mean",
          "ForecastedMortgageValue": "mean",
          "Rent_to_Mortgage_Ratio": "mean"
      })
      .reset_index()
      .sort_values("Rent_to_Mortgage_Ratio", ascending=False)
)

region_summary.head()


In [ ]:
df["Buy_Rent_Index"] = np.where(
    (final_df["Rent_to_Mortgage_Ratio"] > 1.1) & (final_df["mortgage_rate"] < 6),
    "Buy",
    np.where(final_df["Rent_to_Mortgage_Ratio"] < 0.9, "Rent", "Neutral")
)

In [ ]:
decision_summary = df.groupby("StateName")["Buy_Rent_Index"].value_counts(normalize=True).unstack().fillna(0)

In [ ]:
decision_summary